<div style="background-color:#ED1C24; color:white; padding:20px 30px; border-radius:10px;">
<h1 style="color:white; margin:0;">Session 9.1: KNN & Simple Linear Regression</h1>
<p style="color:white; margin:5px 0 0 0;">Practical Lab — Your First Two ML Algorithms</p>
</div>

**Program:** Vishlesan i-Hub IIT Patna x Masai School -- AIM (AI & Machine Learning)
**Session ID:** 9.1 | **Week:** 9
**Prerequisites:** Session 8.2 (ML Pipeline & scikit-learn Introduction)

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain **KNN intuition** — find k nearest neighbors, use their values to predict
2. Compute **Euclidean** and **Manhattan** distance metrics
3. Build **KNN** models using sklearn (classification and regression)
4. Build **simple linear regression** and interpret slope & intercept
5. Evaluate models using **RMSE**, **MAE**, and **R-squared**
6. **Compare** KNN vs Linear Regression — understand when to use which
7. Identify **overfitting** and **underfitting** through the k hyperparameter

## Prerequisites

You should be comfortable with:
- **NumPy** arrays and vectorized operations (Modules 1-2)
- **Pandas** DataFrames and basic plotting (Modules 1-2)
- **Gradient descent** — how models learn parameters (Session 8.1)
- **sklearn API** — `fit()`, `predict()`, `score()`, train/test split, Pipeline (Session 8.2)
- **RMSE and R-squared** basics (introduced in Session 8.2)

---

## Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "colab"
np.random.seed(42)
print("Setup complete! sklearn + Plotly ready.")

Setup complete! sklearn + Plotly ready.


In [2]:
# ============================================================
#  NOTEBOOK FORMATTING HELPERS
# ============================================================

from IPython.display import HTML, display

_BOX_STYLES = {
    "definition": ("#448aff", "#e3f2fd", "#1565c0"),
    "tip":        ("#00c853", "#e8f5e9", "#2e7d32"),
    "warning":    ("#ff9100", "#fff3e0", "#e65100"),
    "danger":     ("#ff1744", "#fce4ec", "#c62828"),
    "math":       ("#7c4dff", "#ede7f6", "#4527a0"),
    "output":     ("#00b8d4", "#e0f7fa", "#006064"),
    "industry":   ("#009688", "#e0f2f1", "#004d40"),
}

In [3]:
def box(kind, title, content):
    """Display a styled callout box."""
    border, bg, title_clr = _BOX_STYLES[kind]
    display(HTML(f"""
    <div style="margin:12px 0; color: grey; padding:12px 16px; border-left:4px solid {border};
                background-color:{bg}; border-radius:4px;">
    <strong style="color:{title_clr};">{title}</strong><br>{content}
    </div>"""))

---

## Bridge from Session 8.2

In Session 8.2, you used **one algorithm** — LinearRegression. It worked. But there are hundreds of ML algorithms. How do you know which one to use?

Today you learn **two** algorithms and, more importantly, how to **compare** them. This is the most important skill in applied ML: model selection.

---

## Business Context: Amazon's Recommendation Engine

Amazon's "Customers who bought this also bought..." feature is essentially **KNN applied to purchase history**. Find customers with similar buying patterns, recommend what similar customers purchased.

This simple algorithm accounts for an estimated **35% of Amazon's total revenue**.

**The lesson:** sometimes the simplest algorithm is the most profitable. But as data grows, you need to compare alternatives. Today you learn that skill.

In [4]:
box("industry", "Amazon's KNN-Based Recommendations",
    "Amazon's item-to-item collaborative filtering finds products that are "
    "'nearest neighbors' in purchase-pattern space. The same KNN intuition "
    "you learn today powered a system generating billions in revenue.")

This session covers two fundamentally different approaches:
- **KNN (Instance-based):** No training. Stores all data. Computes at prediction time.
- **Linear Regression (Parametric):** Learns parameters (slope, intercept) during training. Fast predictions.

By the end, you will build, evaluate, and compare both on the same dataset.

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 1: KNN Intuition — Find Similar, Predict Similar</h2>
</div>

### The Neighborhood Analogy

Moving to a new city and wondering if the neighborhood is safe? Look at the **5 closest houses**. If 4 out of 5 have security cameras and well-maintained gardens, the neighborhood is probably safe.

That is KNN with k=5.

In [5]:
box("definition", "K-Nearest Neighbors (KNN)",
    "To predict for a new data point, find the <b>k closest</b> training points "
    "and use their values.<br><br>"
    "<b>Classification:</b> majority vote of k neighbors (most common class wins).<br>"
    "<b>Regression:</b> average value of k neighbors.<br><br>"
    "<b>No training phase</b> — KNN stores all training data and computes distances "
    "at prediction time. This is called <b>lazy learning</b>.")

 We create a 2D classification dataset: **safe vs unsafe neighborhoods** based on distance to city center and average house price. Hover over each point in the interactive chart below to see its class and feature values.

In [6]:
# Generate 2D classification data
X_safe = np.random.normal([8, 70], [2, 15], (30, 2))
X_unsafe = np.random.normal([3, 30], [1.5, 10], (30, 2))
X_all = np.vstack([X_safe, X_unsafe])
y_all = np.array([1]*30 + [0]*30)
print(f"Generated {len(X_all)} points: 30 Safe, 30 Unsafe")
X_all.shape

Generated 60 points: 30 Safe, 30 Unsafe


(60, 2)

In [7]:
X_all

array([[ 8.99342831, 67.92603548],
       [ 9.29537708, 92.84544785],
       [ 7.53169325, 66.48794565],
       [11.15842563, 81.51152094],
       [ 7.06105123, 78.13840065],
       [ 7.07316461, 63.0140537 ],
       [ 8.48392454, 41.30079633],
       [ 4.55016433, 61.56568706],
       [ 5.97433776, 74.71370999],
       [ 6.18395185, 48.81544448],
       [10.93129754, 66.61335549],
       [ 8.13505641, 48.62877721],
       [ 6.91123455, 71.66383885],
       [ 5.69801285, 75.63547028],
       [ 6.79872262, 65.62459375],
       [ 6.79658678, 97.78417277],
       [ 7.97300555, 54.13433607],
       [ 9.64508982, 51.68734525],
       [ 8.41772719, 40.60494814],
       [ 5.3436279 , 72.95291854],
       [ 9.47693316, 72.57052422],
       [ 7.76870344, 65.48344457],
       [ 5.04295602, 59.20233687],
       [ 7.07872246, 85.85683339],
       [ 8.68723658, 43.55439767],
       [ 8.64816794, 64.22376579],
       [ 6.646156  , 79.17514433],
       [10.06199904, 83.96920179],
       [ 6.32156495,

In [8]:
# INTERACTIVE: Classification data scatter
fig = go.Figure()
for cls, color, name in [(1, "#2ECC71", "Safe"), (0, "#E74C3C", "Unsafe")]:
    mask = y_all == cls
    fig.add_trace(go.Scatter(
        x=X_all[mask, 0], y=X_all[mask, 1], mode="markers",
        marker=dict(color=color, size=10, opacity=0.7),
        name=name,
        hovertemplate="Class: "+name+"<br>Distance: %{x:.1f} km<br>Price: %{y:.1f} Lakhs<extra></extra>"))
fig.update_layout(title="Neighborhood Classification Data", width=800, height=500,
    xaxis_title="Distance to City Center (km)", yaxis_title="House Price (Lakhs)")
fig.show()

### KNN From Scratch

Before using sklearn, let's understand what KNN does step by step. Given a **new point** at (5, 50), we:
1. Compute the Euclidean distance to **every** training point
2. Sort by distance (closest first)
3. Take the **k=5** nearest neighbors
4. Count their classes → **majority vote** = prediction

In [10]:
# KNN from scratch for one point
new_point = np.array([[5, 50]])
distances = np.sqrt(np.sum((X_all - new_point)**2, axis=1))
k = 5
nearest_idx = np.argsort(distances)[:k]
nearest_labels = y_all[nearest_idx]
pred = 1 if nearest_labels.sum() > k/2 else 0
safe_count = int(nearest_labels.sum())
result = "Safe" if pred == 1 else "Unsafe"

In [11]:
print(f"New point: ({new_point[0,0]}, {new_point[0,1]})")
print(f"\n5 Nearest Neighbors:")
for j, idx in enumerate(nearest_idx):
    cls = "Safe" if y_all[idx] == 1 else "Unsafe"
    print(f"  #{j+1}: ({X_all[idx,0]:.1f}, {X_all[idx,1]:.1f}) — {cls} — dist={distances[idx]:.2f}")
print(f"\nVote: {safe_count} Safe, {k - safe_count} Unsafe")
print(f"Prediction: {result}")

New point: (5, 50)

5 Nearest Neighbors:
  #1: (6.2, 48.8) — Safe — dist=1.67
  #2: (8.1, 48.6) — Safe — dist=3.42
  #3: (2.9, 45.6) — Unsafe — dist=4.81
  #4: (3.5, 45.4) — Unsafe — dist=4.84
  #5: (9.6, 51.7) — Safe — dist=4.94

Vote: 3 Safe, 2 Unsafe
Prediction: Safe


The interactive chart below shows the new point (gold star) and its 5 nearest neighbors connected by dashed lines. Hover over any neighbor line to see the distance.

In [12]:
# INTERACTIVE: KNN Neighborhood Visualizer
fig = go.Figure()
for cls, color, name in [(1, "#2ECC71", "Safe"), (0, "#E74C3C", "Unsafe")]:
    mask = y_all == cls
    fig.add_trace(go.Scatter(x=X_all[mask,0], y=X_all[mask,1], mode="markers",
        marker=dict(color=color, size=10, opacity=0.7), name=name,
        hovertemplate=name+": (%{x:.1f}, %{y:.1f})<extra></extra>"))
fig.add_trace(go.Scatter(x=[new_point[0,0]], y=[new_point[0,1]], mode="markers",
    marker=dict(color="gold", size=18, symbol="star", line=dict(color="black", width=1)),
    name="New Point"))

In [13]:
for idx in nearest_idx:
    cls = "Safe" if y_all[idx]==1 else "Unsafe"
    fig.add_trace(go.Scatter(x=[new_point[0,0], X_all[idx,0]], y=[new_point[0,1], X_all[idx,1]],
        mode="lines", line=dict(dash="dash", color="gray", width=1), showlegend=False,
        hovertemplate=f"Neighbor: {cls}, dist={distances[idx]:.2f}<extra></extra>"))
fig.update_layout(title=f"KNN (k=5): Prediction = {result}", width=800, height=500,
    xaxis_title="Distance to Center (km)", yaxis_title="Price (Lakhs)")
fig.show()

In [14]:
box("output", "KNN Prediction Result",
    f"The 5 nearest neighbors voted: {int(safe_count)} Safe, {k-int(safe_count)} Unsafe. "
    f"Prediction: <b>{result}</b>.<br><br>"
    "KNN does NOT learn parameters — it memorizes training data and computes distances "
    "at prediction time. This is why it is called a <b>lazy learner</b>.")

### Distance Metrics — How Do We Measure "Close"?

Two main distance formulas:

**Euclidean Distance** (straight-line): $d = \sqrt{\sum(x_i - y_i)^2}$

**Manhattan Distance** (city-block): $d = \sum|x_i - y_i|$

Euclidean is the default for sklearn KNN. Manhattan is better when features are on different scales or outliers are present.

In [15]:
box("definition", "Distance Metrics",
    "<b>Euclidean:</b> straight-line distance = <code>np.linalg.norm(a - b)</code> from Session 6.1<br>"
    "<b>Manhattan:</b> sum of absolute differences = walking on a city grid<br><br>"
    "<b>Default for sklearn KNN:</b> Euclidean (metric='minkowski', p=2)")

The chart below shows both distances between two points. The **blue diagonal** is Euclidean (straight-line). The **orange L-shape** is Manhattan (grid path). Hover to see the computed values.

In [16]:
# INTERACTIVE: Euclidean vs Manhattan
A, B = np.array([2, 3]), np.array([7, 8])
euc = np.linalg.norm(A - B)
man = np.sum(np.abs(A - B))

fig = go.Figure()
fig.add_trace(go.Scatter(x=[A[0], B[0]], y=[A[1], B[1]], mode="markers+text",
    marker=dict(size=14, color=["#3498DB", "#E74C3C"]),
    text=["A (2,3)", "B (7,8)"], textposition="top center", showlegend=False))
fig.add_trace(go.Scatter(x=[A[0], B[0]], y=[A[1], B[1]], mode="lines",
    line=dict(color="#3498DB", width=3), name=f"Euclidean = {euc:.2f}"))
fig.add_trace(go.Scatter(x=[A[0], B[0], B[0]], y=[A[1], A[1], B[1]], mode="lines",
    line=dict(color="#FF9100", width=3, dash="dash"), name=f"Manhattan = {man:.1f}"))
fig.update_layout(title="Euclidean vs Manhattan Distance", width=700, height=450,
    xaxis=dict(range=[0, 10], dtick=1), yaxis=dict(range=[0, 10], dtick=1))
fig.show()

In [17]:
box("tip", "Connection to Session 6.1",
    "<code>np.linalg.norm(a - b)</code> IS Euclidean distance. "
    "KNN uses the same vector operations from linear algebra.<br>"
    "Manhattan distance = <code>np.sum(np.abs(a - b))</code>.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 2: KNN with sklearn</h2>
</div>

In [18]:
box("math", "sklearn KNN API",
    "<code>KNeighborsClassifier(n_neighbors=5)</code> — create classifier<br>"
    "<code>model.fit(X_train, y_train)</code> — stores the training data (no learning!)<br>"
    "<code>model.predict(X_test)</code> — computes distances to all training points<br>"
    "<code>model.score(X_test, y_test)</code> — returns accuracy (classification) or R&sup2; (regression)")

Split the neighborhood data into train/test sets and fit a KNN classifier with k=5:

In [19]:
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_tr, y_tr)
acc = knn.score(X_te, y_te)
print(f"KNN Classifier (k=5)")
print(f"Training samples: {len(X_tr)}")
print(f"Test samples:     {len(X_te)}")
print(f"Accuracy:         {acc:.1%}")

KNN Classifier (k=5)
Training samples: 48
Test samples:     12
Accuracy:         91.7%


How does the choice of **k** affect accuracy? The following loop tests multiple values of k:

In [23]:
k_values = [1, 3, 5, 7, 11, 15]
accuracies = []
print(f"{'k':>3} {'Accuracy':>10}")
print("-" * 15)
for k_val in k_values:
    a = KNeighborsClassifier(n_neighbors=k_val).fit(X_tr, y_tr).score(X_te, y_te)
    accuracies.append(a)
    print(f"{k_val:>3} {a:>10.1%}")

  k   Accuracy
---------------
  1     100.0%
  3     100.0%
  5      91.7%
  7      91.7%
 11      91.7%
 15     100.0%


The interactive bar chart below highlights the best k value. Hover over each bar to see the exact accuracy.

In [24]:
# INTERACTIVE: Accuracy vs k
best_idx = np.argmax(accuracies)
colors = ["#3498DB" if i != best_idx else "#2ECC71" for i in range(len(k_values))]
fig = go.Figure(go.Bar(x=[str(k) for k in k_values], y=accuracies,
    marker_color=colors, hovertemplate="k=%{x}<br>Accuracy=%{y:.1%}<extra></extra>"))
fig.update_layout(title="Effect of k on KNN Accuracy", width=700, height=400,
    xaxis_title="k (number of neighbors)", yaxis_title="Accuracy",
    yaxis=dict(tickformat=".0%"))
fig.show()

In [25]:
box("warning", "Scaling is ESSENTIAL for KNN",
    "KNN uses distances. If Feature A ranges 0-1 and Feature B ranges 0-10000, "
    "Feature B dominates ALL distance calculations.<br><br>"
    "<b>Always use StandardScaler in a Pipeline with KNN.</b><br>"
    "Linear Regression adjusts coefficients internally, but KNN cannot.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 3: Simple Linear Regression — The Line of Best Fit</h2>
</div>

### Which Line is "Best"?

Given scattered data points, infinitely many lines could be drawn through them. Which line is "best"? How do we define "best"?

**Answer:** The line that **minimizes the total prediction error** (Mean Squared Error).

The following code generates a dataset with a known true relationship: **y = 3.5x + 8 + noise**. This is regression data — continuous target, one feature.

In [26]:
# Generate regression data with known relationship
np.random.seed(42)
X_reg = np.random.uniform(1, 10, 100).reshape(-1, 1)
y_reg = 3.5 * X_reg.flatten() + 8 + np.random.normal(0, 3, 100)
x_flat = X_reg.flatten()

print(f"Dataset: {len(y_reg)} points")
print(f"True relationship: y = 3.5x + 8 + noise")
print(f"X range: [{x_flat.min():.1f}, {x_flat.max():.1f}]")
print(f"y range: [{y_reg.min():.1f}, {y_reg.max():.1f}]")

Dataset: 100 points
True relationship: y = 3.5x + 8 + noise
X range: [1.0, 9.9]
y range: [10.2, 44.9]


Before fitting the best line, consider these **four candidate lines**. Each makes different predictions. Which one has the lowest error? Hover over each line to see its MSE.

In [27]:
# Define 4 candidate lines and compute MSE for each
lr_fit = LinearRegression().fit(X_reg, y_reg)
candidates = [
    (1.0, 20.0, "#E74C3C", "Random Guess"),
    (2.5, 12.0, "#FF9100", "Closer"),
    (3.5,  8.0, "#9B59B6", "True Line"),
    (lr_fit.coef_[0], lr_fit.intercept_, "#2ECC71", "Best Fit (sklearn)"),
]
print(f"{'Line':<20} {'Equation':<18} {'MSE':>8}")
print("-" * 48)
for m, b, _, name in candidates:
    mse = np.mean((y_reg - (m * x_flat + b))**2)
    print(f"{name:<20} y={m:.1f}x+{b:.1f}{'':>5} {mse:>8.2f}")

Line                 Equation                MSE
------------------------------------------------
Random Guess         y=1.0x+20.0         47.50
Closer               y=2.5x+12.0         13.86
True Line            y=3.5x+8.0          7.43
Best Fit (sklearn)   y=3.3x+8.8          7.26


In [28]:
# INTERACTIVE: "Why This Line?" Explorer
xl = np.linspace(0.5, 10.5, 100)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_flat, y=y_reg, mode="markers",
    marker=dict(color="#95A5A6", size=7, opacity=0.6), name="Data",
    hovertemplate="x=%{x:.1f}, y=%{y:.1f}<extra></extra>"))
for m, b, color, name in candidates:
    mse = np.mean((y_reg - (m * x_flat + b))**2)
    fig.add_trace(go.Scatter(x=xl, y=m*xl+b, mode="lines",
        line=dict(color=color, width=2), name=f"{name} (MSE={mse:.1f})"))

In [29]:
# Add residual lines from data to best-fit line
best_m, best_b = lr_fit.coef_[0], lr_fit.intercept_
for xi, yi in zip(x_flat[::5], y_reg[::5]):
    yp = best_m * xi + best_b
    fig.add_trace(go.Scatter(x=[xi, xi], y=[yi, yp], mode="lines",
        line=dict(color="#2ECC71", width=0.8), showlegend=False, opacity=0.4,
        hovertemplate=f"Error: {yi-yp:+.1f}<extra></extra>"))
fig.update_layout(title="Which Line Fits Best? (Green = lowest MSE)", width=900, height=500)
fig.show()

In [30]:
box("definition", "Mean Squared Error (MSE)",
    "MSE = (1/n) &Sigma;(y<sub>i</sub> - y&#770;<sub>i</sub>)&sup2;<br><br>"
    "The line with the <b>LOWEST MSE</b> is the 'best fit'. Squaring ensures "
    "positive errors don't cancel negative errors, and penalizes large errors more.")

sklearn's `LinearRegression` finds this best-fit line automatically using gradient descent (from Session 8.1). One line of code replaces the manual search above:

In [31]:
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
lr = LinearRegression().fit(Xr_tr, yr_tr)
yr_pred = lr.predict(Xr_te)

print(f"Learned equation: y = {lr.coef_[0]:.2f}x + {lr.intercept_:.2f}")
print(f"True equation:    y = 3.50x + 8.00")
print(f"RMSE: {np.sqrt(mean_squared_error(yr_te, yr_pred)):.2f}")
print(f"R-squared: {r2_score(yr_te, yr_pred):.4f}")

Learned equation: y = 3.37x + 8.56
True equation:    y = 3.50x + 8.00
RMSE: 2.43
R-squared: 0.9354


The interactive scatter below shows the test data with the fitted regression line. Hover over any data point to see its actual value, predicted value, and residual (error).

In [32]:
# INTERACTIVE: Regression line with residual hover
fig = go.Figure()
residuals = yr_te - yr_pred
hover_text = [f"Actual: {a:.1f}<br>Predicted: {p:.1f}<br>Residual: {a-p:+.1f}"
              for a, p in zip(yr_te, yr_pred)]
fig.add_trace(go.Scatter(x=Xr_te.flatten(), y=yr_te, mode="markers",
    marker=dict(color=residuals, colorscale="RdYlGn", size=9, colorbar=dict(title="Residual")),
    text=hover_text, hovertemplate="%{text}<extra></extra>", name="Test Data"))
xl2 = np.linspace(Xr_te.min(), Xr_te.max(), 100)
fig.add_trace(go.Scatter(x=xl2, y=lr.coef_[0]*xl2+lr.intercept_, mode="lines",
    line=dict(color="#ED1C24", width=2), name=f"y={lr.coef_[0]:.2f}x+{lr.intercept_:.2f}"))
r2 = r2_score(yr_te, yr_pred)
fig.update_layout(title=f"Linear Regression — R-squared={r2:.3f}", width=800, height=500,
    xaxis_title="X", yaxis_title="y")
fig.show()

In [33]:
box("output", "Interpreting the Coefficients",
    f"<b>Slope = {lr.coef_[0]:.2f}:</b> for each 1-unit increase in X, y increases by {lr.coef_[0]:.2f} units.<br>"
    f"<b>Intercept = {lr.intercept_:.2f}:</b> the predicted y when X = 0.<br><br>"
    "True relationship: y = 3.5x + 8. The model recovered approximately the correct parameters.")

**Connection to Session 8.1:** when you call `model.fit()`, sklearn runs gradient descent internally — the same algorithm you coded from scratch. The slope and intercept are the parameters that gradient descent optimized to minimize MSE.

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 4: Evaluation Metrics Deep Dive</h2>
</div>

In [34]:
box("definition", "The Four Key Metrics",
    "<b>Accuracy</b> (classification): correct predictions / total predictions<br>"
    "<b>RMSE</b> (regression): &radic;(mean of squared errors) — error in original units<br>"
    "<b>MAE</b> (regression): mean of absolute errors — robust to outliers<br>"
    "<b>R&sup2;</b> (regression): fraction of variance explained. 1.0 = perfect, 0 = mean baseline")

### RMSE vs MAE

- **RMSE** penalizes large errors more (because of squaring). Use when large errors are catastrophic.
- **MAE** treats all errors equally. Use when outliers should not dominate.

### Step-by-Step: Computing RMSE Manually

In [35]:
# Step-by-step RMSE computation
residuals = yr_te - yr_pred
squared = residuals ** 2
mse = np.mean(squared)
rmse = np.sqrt(mse)
print("=== RMSE Step by Step ===")
print(f"1. Residuals (first 5):  {residuals[:5].round(2)}")
print(f"2. Squared residuals:    {squared[:5].round(2)}")
print(f"3. Mean of squared:      {mse:.4f}")
print(f"4. Square root (RMSE):   {rmse:.4f}")
print(f"\nsklearn check:           {np.sqrt(mean_squared_error(yr_te, yr_pred)):.4f}")

=== RMSE Step by Step ===
1. Residuals (first 5):  [ 1.62 -0.86 -3.21 -2.36  2.26]
2. Squared residuals:    [ 2.62  0.74 10.3   5.57  5.09]
3. Mean of squared:      5.8833
4. Square root (RMSE):   2.4256

sklearn check:           2.4256


### Step-by-Step: Computing MAE Manually

In [36]:
# Step-by-step MAE computation
abs_errors = np.abs(residuals)
mae = np.mean(abs_errors)
print("=== MAE Step by Step ===")
print(f"1. Absolute errors (first 5): {abs_errors[:5].round(2)}")
print(f"2. Mean of absolute:          {mae:.4f}")
print(f"\nsklearn check:                {mean_absolute_error(yr_te, yr_pred):.4f}")
print(f"\nRMSE ({rmse:.2f}) >= MAE ({mae:.2f}) — always true (squaring amplifies large errors)")

=== MAE Step by Step ===
1. Absolute errors (first 5): [1.62 0.86 3.21 2.36 2.26]
2. Mean of absolute:          1.7740

sklearn check:                1.7740

RMSE (2.43) >= MAE (1.77) — always true (squaring amplifies large errors)


### R-squared: What Does It LOOK Like?

R-squared measures how much of the variance in y is explained by the model. But what does R-squared = 0.1 vs 0.5 vs 0.9 vs 1.0 actually **look like**?

The interactive chart below answers this question. Each panel shows a different noise level, producing different R-squared values.

In [37]:
# Helper function for R-squared Explorer
def make_r2_data(noise_std, seed=0):
    np.random.seed(seed)
    x = np.random.uniform(1, 10, 60)
    y = 2 * x + 5 + np.random.normal(0, noise_std, 60)
    mdl = LinearRegression().fit(x.reshape(-1,1), y)
    y_pred = mdl.predict(x.reshape(-1,1))
    return x, y, y_pred, r2_score(y, y_pred), mdl

In [38]:
# Generate 4 datasets with increasing noise
datasets = [
    make_r2_data(8.0, seed=10),   # low R2
    make_r2_data(4.0, seed=20),   # medium R2
    make_r2_data(1.5, seed=30),   # high R2
    make_r2_data(0.01, seed=40),  # perfect R2
]
for x, y, yp, r2v, _ in datasets:
    print(f"Noise std={np.std(y-yp):.1f} -> R-squared={r2v:.3f}")

Noise std=8.3 -> R-squared=0.221
Noise std=4.0 -> R-squared=0.540
Noise std=1.4 -> R-squared=0.935
Noise std=0.0 -> R-squared=1.000


In [39]:
# INTERACTIVE: R-squared Explorer (SIGNATURE VISUALIZATION)
fig = make_subplots(rows=1, cols=4, subplot_titles=[f"R-squared = {d[3]:.2f}" for d in datasets])
for i, (x, y, yp, r2v, mdl) in enumerate(datasets):
    col = i + 1
    fig.add_trace(go.Scatter(x=x, y=y, mode="markers", marker=dict(color="#3498DB", size=6),
        showlegend=False, hovertemplate="Actual: %{y:.1f}<extra></extra>"), row=1, col=col)
    xl3 = np.linspace(x.min(), x.max(), 50)
    fig.add_trace(go.Scatter(x=xl3, y=mdl.predict(xl3.reshape(-1,1)), mode="lines",
        line=dict(color="#ED1C24", width=2), showlegend=False), row=1, col=col)
    for xj, yj, ypj in zip(x[::3], y[::3], yp[::3]):
        fig.add_trace(go.Scatter(x=[xj,xj], y=[yj,ypj], mode="lines",
            line=dict(color="#ED1C24", width=0.5), showlegend=False, opacity=0.3), row=1, col=col)
fig.update_layout(title="R-squared Explorer: What Does Model Fit LOOK Like?",
    width=1100, height=380, showlegend=False)
fig.show()

In [40]:
box("math", "R-squared Formula",
    "R&sup2; = 1 - (SS<sub>res</sub> / SS<sub>tot</sub>)<br><br>"
    "SS<sub>res</sub> = &Sigma;(y<sub>i</sub> - y&#770;<sub>i</sub>)&sup2; (sum of squared residuals — model errors)<br>"
    "SS<sub>tot</sub> = &Sigma;(y<sub>i</sub> - y&#772;)&sup2; (total variance — spread around the mean)<br><br>"
    "R&sup2; = 1 → SS<sub>res</sub> = 0 (perfect predictions). "
    "R&sup2; = 0 → SS<sub>res</sub> = SS<sub>tot</sub> (no better than predicting the mean).")

### Can R-squared Be Negative?

**Yes.** If the model's predictions are **worse** than simply predicting the mean for every point, R-squared is negative. This means the model is fundamentally wrong for this data.

In [41]:
# Demonstrate negative R-squared
np.random.seed(42)
x_neg = np.array([1, 2, 3, 4, 5])
y_neg = np.array([2, 4, 5, 4, 5])
y_bad_pred = np.array([10, 10, 10, 10, 10])  # terrible predictions
y_mean_pred = np.full(5, y_neg.mean())         # just predict the mean

r2_bad = r2_score(y_neg, y_bad_pred)
r2_mean = r2_score(y_neg, y_mean_pred)
print(f"Bad model predictions:  {y_bad_pred} → R-squared = {r2_bad:.2f}")
print(f"Mean prediction:        {y_mean_pred} → R-squared = {r2_mean:.2f}")
print(f"\nR-squared < 0 means WORSE than predicting the mean!")

Bad model predictions:  [10 10 10 10 10] → R-squared = -30.00
Mean prediction:        [4. 4. 4. 4. 4.] → R-squared = 0.00

R-squared < 0 means WORSE than predicting the mean!


In [42]:
box("danger", "R-squared < 0 Means Worse Than the Mean!",
    "A negative R-squared means your model is WORSE than predicting the average for every point. "
    "This usually means: (1) the model is fundamentally wrong for this data, or "
    "(2) you are evaluating on very different data than you trained on.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 5: Model Comparison — KNN vs Linear Regression</h2>
</div>

The real skill in ML is not building ONE model — it is comparing **multiple models** on the same data and choosing the best one for your situation.

In [43]:
# Fit KNN Regressor on the same regression data
knn_reg = KNeighborsRegressor(n_neighbors=5).fit(Xr_tr, yr_tr)
yr_knn = knn_reg.predict(Xr_te)

print("=== MODEL COMPARISON ===\n")
metrics = [("RMSE", lambda y,p: np.sqrt(mean_squared_error(y,p))),
           ("MAE", mean_absolute_error), ("R-squared", r2_score)]
lr_label, knn_label = "Linear Reg", "KNN (k=5)"
print(f"{'Metric':<12} {lr_label:>12} {knn_label:>12} {'Winner':>10}")
print("-" * 48)
for name, fn in metrics:
    v1, v2 = fn(yr_te, yr_pred), fn(yr_te, yr_knn)
    better = "higher" if name == "R-squared" else "lower"
    winner = lr_label if (v1>v2 if better=="higher" else v1<v2) else knn_label
    print(f"{name:<12} {v1:>12.4f} {v2:>12.4f} {winner:>10}")

=== MODEL COMPARISON ===

Metric         Linear Reg    KNN (k=5)     Winner
------------------------------------------------
RMSE               2.4256       2.8244 Linear Reg
MAE                1.7740       2.2595 Linear Reg
R-squared          0.9354       0.9124 Linear Reg


The interactive chart below shows **Actual vs Predicted** for both models side by side. Points on the red dashed diagonal represent perfect predictions. Hover to see individual errors.

In [44]:
# INTERACTIVE: Side-by-side Actual vs Predicted
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"Linear Regression (R-sq={r2_score(yr_te,yr_pred):.3f})",
    f"KNN k=5 (R-sq={r2_score(yr_te,yr_knn):.3f})"])
for col, preds, name in [(1, yr_pred, "LR"), (2, yr_knn, "KNN")]:
    hover = [f"Actual: {a:.1f}<br>Pred: {p:.1f}<br>Error: {a-p:+.1f}" for a,p in zip(yr_te, preds)]
    fig.add_trace(go.Scatter(x=yr_te, y=preds, mode="markers", marker=dict(size=8, opacity=0.7),
        text=hover, hovertemplate="%{text}<extra></extra>", showlegend=False), row=1, col=col)
    rng = [min(yr_te.min(), preds.min()), max(yr_te.max(), preds.max())]
    fig.add_trace(go.Scatter(x=rng, y=rng, mode="lines",
        line=dict(color="red", dash="dash", width=1), showlegend=False), row=1, col=col)
fig.update_layout(title="Model Showdown: Actual vs Predicted", width=1000, height=450)
fig.show()

### Decision Guide: KNN vs Linear Regression

| Aspect | KNN | Linear Regression |
|--------|-----|------------------|
| **Assumption** | None about data shape | Assumes LINEAR relationship |
| **Training** | None (stores all data) | Learns m, b via gradient descent |
| **Prediction speed** | SLOW (computes distances) | FAST (just mx+b) |
| **Interpretability** | Low ("neighbors said so") | High ("each unit of X adds m") |
| **Non-linear data** | Handles naturally | Fails (need polynomial features) |
| **Scaling needed?** | YES (essential) | Not required (but helps) |
| **Memory** | Stores all training data | Stores only m and b |

In [45]:
box("tip", "When to Use Which",
    "<b>Use KNN when:</b> data is non-linear, dataset is small-to-medium, interpretability not needed.<br>"
    "<b>Use Linear Regression when:</b> relationship is approximately linear, large dataset, "
    "interpretability is important, prediction speed matters.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 6: The k Hyperparameter</h2>
</div>

In [46]:
box("definition", "Hyperparameter vs Parameter",
    "A <b>hyperparameter</b> is a setting YOU choose BEFORE training (e.g., k in KNN). "
    "It is NOT learned from data.<br>"
    "A <b>parameter</b> is learned FROM data (e.g., slope and intercept in linear regression).<br><br>"
    "Finding the best hyperparameter is called <b>hyperparameter tuning</b>.")

The following code trains KNN Regressor for **k=1 through k=20**, recording both train and test R-squared. The gap between them reveals a critical phenomenon.

In [47]:
k_range = range(1, 21)
train_r2 = []
test_r2 = []
for k_val in k_range:
    knn_k = KNeighborsRegressor(n_neighbors=k_val).fit(Xr_tr, yr_tr)
    train_r2.append(knn_k.score(Xr_tr, yr_tr))
    test_r2.append(knn_k.score(Xr_te, yr_te))
best_k = list(k_range)[np.argmax(test_r2)]
print(f"Best k: {best_k} (test R-squared = {max(test_r2):.4f})")

Best k: 2 (test R-squared = 0.9316)


The interactive chart below shows how k affects performance. At **small k** (left), the model overfits. At **large k** (right), it underfits. Hover to see exact values.

In [48]:
# INTERACTIVE: k-Effect Explorer
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(k_range), y=train_r2, mode="lines+markers",
    name="Train R-squared", line=dict(color="#3498DB"), marker=dict(size=6),
    hovertemplate="k=%{x}<br>Train R-sq=%{y:.4f}<extra></extra>"))
fig.add_trace(go.Scatter(x=list(k_range), y=test_r2, mode="lines+markers",
    name="Test R-squared", line=dict(color="#E74C3C"), marker=dict(size=6),
    hovertemplate="k=%{x}<br>Test R-sq=%{y:.4f}<extra></extra>"))
fig.add_vline(x=best_k, line=dict(color="#2ECC71", dash="dash", width=2),
    annotation_text=f"Best k={best_k}", annotation_position="top right")

In [49]:
fig.add_vrect(x0=0.5, x1=3.5, fillcolor="#FADBD8", opacity=0.2,
    line_width=0, annotation_text="Overfit zone", annotation_position="top left")
fig.add_vrect(x0=14.5, x1=20.5, fillcolor="#D6EAF8", opacity=0.2,
    line_width=0, annotation_text="Underfit zone", annotation_position="top right")
fig.update_layout(title="Effect of k on KNN Performance", width=900, height=500,
    xaxis_title="k (number of neighbors)", yaxis_title="R-squared")
fig.show()

### Overfitting and Underfitting — First Look

These are two of the most important concepts in all of machine learning:

**Overfitting** (k too small, e.g., k=1):
- The model **memorizes** training data instead of learning general patterns
- **Symptom:** train R-squared is very high (even 1.0) but test R-squared is much lower
- At k=1, each point is its own nearest neighbor, so the model perfectly "predicts" every training point

**Underfitting** (k too large, e.g., k=20):
- The model is **too simple** to capture the real pattern — it averages too many neighbors
- **Symptom:** BOTH train and test R-squared are low
- The model is not wrong — it is just too blunt

**Goal:** Find the **sweet spot** (optimal k) where test performance is highest.

In [50]:
box("warning", "Overfitting vs Underfitting (First Look)",
    "These two errors are the central tension in ALL of machine learning:<br><br>"
    "<b>Overfitting (k too small):</b> memorizes noise — great on training data, "
    "poor on new data.<br>"
    "<b>Underfitting (k too large):</b> too simple — poor on everything.<br>"
    "<b>Goal:</b> find the sweet spot (optimal k) where test performance peaks.<br><br>"
    "The formal theory behind this (bias-variance tradeoff) is covered later in AIM201. "
    "For now: if train score >> test score, you are overfitting.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 7: Mini-Project — Model Showdown</h2>
</div>

Build both KNN Regressor and Linear Regression on a dataset with **non-linear** components. Compare metrics, visualize predictions, and recommend which model to deploy.

In [51]:
# Generate complex dataset with non-linear component
np.random.seed(42)
n_mp = 200
X_mp = np.random.uniform(0, 10, (n_mp, 3))
y_mp = 5*X_mp[:,0] + 3*np.sin(X_mp[:,1]*2) + 2*X_mp[:,2] + np.random.normal(0, 2, n_mp)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(X_mp, y_mp, test_size=0.2, random_state=42)
print(f"Dataset: {n_mp} points, 3 features")
print(f"Hidden relationship: y = 5*x1 + 3*sin(2*x2) + 2*x3 + noise")
print(f"Train: {len(Xm_tr)}, Test: {len(Xm_te)}")

Dataset: 200 points, 3 features
Hidden relationship: y = 5*x1 + 3*sin(2*x2) + 2*x3 + noise
Train: 160, Test: 40


Fit both models using **Pipelines** (StandardScaler + model) to ensure fair comparison:

In [52]:
# Pipeline for each model
pipe_lr = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
pipe_knn = Pipeline([("scaler", StandardScaler()), ("model", KNeighborsRegressor(n_neighbors=7))])
pipe_lr.fit(Xm_tr, ym_tr)
pipe_knn.fit(Xm_tr, ym_tr)
ym_lr = pipe_lr.predict(Xm_te)
ym_knn = pipe_knn.predict(Xm_te)
print("Both models trained!")

Both models trained!


In [53]:
# Metrics comparison
print("=" * 55)
print("MINI-PROJECT: MODEL SHOWDOWN")
print("=" * 55)
print(f"\n{'Metric':<12} {'Linear Reg':>12} {'KNN (k=7)':>12} {'Winner':>10}")
print("-" * 48)
for name, fn in [("RMSE", lambda y,p: np.sqrt(mean_squared_error(y,p))),
                  ("MAE", mean_absolute_error), ("R-squared", r2_score)]:
    v1, v2 = fn(ym_te, ym_lr), fn(ym_te, ym_knn)
    better_fn = max if name == "R-squared" else min
    w = "Linear Reg" if better_fn(v1,v2) == v1 else "KNN (k=7)"
    print(f"{name:<12} {v1:>12.4f} {v2:>12.4f} {w:>10}")

MINI-PROJECT: MODEL SHOWDOWN

Metric         Linear Reg    KNN (k=7)     Winner
------------------------------------------------
RMSE               2.7053       4.3883 Linear Reg
MAE                2.1416       3.5543 Linear Reg
R-squared          0.9749       0.9338 Linear Reg


In [54]:
# INTERACTIVE: Mini-project Actual vs Predicted
r2_lr = r2_score(ym_te, ym_lr)
r2_knn = r2_score(ym_te, ym_knn)
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"Linear Regression (R-sq={r2_lr:.3f})", f"KNN k=7 (R-sq={r2_knn:.3f})"])
for col, preds in [(1, ym_lr), (2, ym_knn)]:
    hover = [f"Actual: {a:.1f}<br>Pred: {p:.1f}<br>Error: {a-p:+.1f}" for a,p in zip(ym_te, preds)]
    fig.add_trace(go.Scatter(x=ym_te, y=preds, mode="markers", marker=dict(size=7, opacity=0.6),
        text=hover, hovertemplate="%{text}<extra></extra>", showlegend=False), row=1, col=col)
    rng = [min(ym_te.min(), preds.min()), max(ym_te.max(), preds.max())]
    fig.add_trace(go.Scatter(x=rng, y=rng, mode="lines",
        line=dict(color="red", dash="dash"), showlegend=False), row=1, col=col)
fig.update_layout(title="Model Showdown: Which Wins on Non-Linear Data?", width=1000, height=450)
fig.show()

**Why KNN wins here:** The dataset has a **non-linear** component (`sin(2*x2)`). KNN captures non-linear patterns naturally because it makes **no assumption** about data shape — it just finds similar points. Linear Regression assumes a straight-line relationship and cannot model the sine curve.

This is why model comparison matters. **No single algorithm is best for all data.**

In [55]:
box("tip", "Model Selection is an Ongoing Process",
    "Amazon started with KNN for recommendations. As data grew, they added matrix factorization, "
    "deep learning, and hybrid approaches. No single model is best forever — the right choice "
    "depends on your <b>data</b>, <b>goals</b>, and <b>constraints</b>.")

---

## Exercise: Self-Check Questions

Test your understanding:

1. If your KNN model has train accuracy=100% but test accuracy=60%, what is happening and how do you fix it?
2. What is the key difference between a hyperparameter and a parameter?
3. If R-squared = -0.3, what does that mean in plain English?
4. For a dataset with 10 million rows, would you choose KNN or Linear Regression for fast predictions? Why?
5. Can KNN be used for both classification AND regression?

<details>
<summary><b>Click for answers</b></summary>

1. **Overfitting** — the model memorized training data. Fix: increase k (use more neighbors).
2. **Hyperparameter** (like k) is set by YOU before training. **Parameter** (like slope) is learned FROM data during training.
3. The model is **worse than just predicting the mean**. The model is fundamentally wrong for this data.
4. **Linear Regression** — it only computes mx+b (instant). KNN must compute distances to ALL training points (very slow on 10M rows).
5. **Yes!** KNeighborsClassifier (majority vote) and KNeighborsRegressor (average value).

</details>

---

## Summary & Key Takeaways

<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 15px; margin: 15px 0;">
<strong>Key Takeaways:</strong>
<ol>
<li><strong>KNN:</strong> Find k nearest neighbors → majority vote (classification) or average (regression)</li>
<li><strong>Euclidean distance:</strong> straight-line = np.linalg.norm(a-b). <strong>Manhattan:</strong> sum of absolute differences</li>
<li><strong>KNN is lazy:</strong> no training phase — stores all data, computes at prediction time</li>
<li><strong>Linear Regression:</strong> y = mx + b. Slope = effect per unit. Intercept = baseline</li>
<li><strong>MSE:</strong> mean of squared errors — the loss function Linear Regression minimizes</li>
<li><strong>RMSE:</strong> square root of MSE — error in original units. <strong>MAE:</strong> mean absolute error — robust to outliers</li>
<li><strong>R-squared:</strong> fraction of variance explained. 1.0 = perfect. 0 = mean baseline. Can be negative</li>
<li><strong>KNN vs LR:</strong> KNN handles non-linear data but is slow. LR is fast and interpretable but assumes linearity</li>
<li><strong>Scaling matters for KNN</strong> (distances are scale-sensitive). Always use StandardScaler in a Pipeline</li>
<li><strong>Overfitting</strong> (small k) = memorizes noise. <strong>Underfitting</strong> (large k) = too simple. Find the sweet spot</li>
</ol>
</div>

**Next Session:** Session 9.2 — Logistic Regression & Classification. If today's question was "what is the VALUE?" (regression), 9.2's question is "which CATEGORY?" (classification). You will learn the confusion matrix, precision, recall, and ROC-AUC.

---
<div style="text-align:center; color:#666; padding:20px;"><em>Vishlesan i-Hub IIT Patna x Masai School</em></div>